## Testing additional length-based metrics

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../Datasets/Development_data.csv', encoding='utf-8')
len(df)

9068

Removing one- and two-sentence utterances to not influence the aggregation calculations 

In [3]:
sentence_count = df.groupby('ID').size()

one_sentence = (sentence_count == 1).sum()
two_sentence = (sentence_count == 2).sum()

print('No. of one-sentence utterances: ', one_sentence)
print('No. of one-sentence utterances: ', two_sentence)

No. of one-sentence utterances:  107
No. of one-sentence utterances:  145


In [4]:
df1 = df[df["ID"].isin(sentence_count[sentence_count >= 3].index)]
len(df1)

8671

In [5]:
check = df1['ID'].unique()
len(check)

548

### Sentence-count based avg

In [6]:
df1['count'] = df1.groupby('ID')['sent_id'].transform('count')
df1.head()


/var/folders/l_/t5l44v1s7d9168_x8bm5ly0h0000gn/T/ipykernel_74627/1040489154.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['count'] = df1.groupby('ID')['sent_id'].transform('count')


,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata,count
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.0,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,2.0,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.0,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3


In [7]:
#df1["annotation_sent"] = df1["annotation_sent"].clip(0, 5)
#df1.sort_values(by="annotation_sent")



In [8]:
count_avg = (
    df1.groupby('ID')
    .apply(lambda x: (x['annotation_sent'] * x['count']).sum() / x['count'].sum())
    .reset_index(name='count_avg')
)

df1 = df1.merge(count_avg, on='ID')
df1.head()

/var/folders/l_/t5l44v1s7d9168_x8bm5ly0h0000gn/T/ipykernel_74627/209540263.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df1.groupby('ID')


,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata,count,count_avg
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.838881
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.0,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.838881
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,2.0,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.838881
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,3.491818
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.0,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,3.491818


In [9]:
df1["count_avg"] = df1["count_avg"].round(2)
df_count = df1.drop_duplicates(subset=['ID']).reset_index()
df_count


,index,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata,count,count_avg
0,0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.84
1,3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,3.49
2,6,ParlaMint-SI_2011-06-21-SDZ5-Redna-29.ana.u178,ParlaMint-SI_2011-06-21-SDZ5-Redna-29.ana.seg6...,"Hvala lepa. V bistvu se strinjam z vami, gospo...",Hvala lepa.,Thank you very much.,False,Positive,5.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-21-SDZ5-R...,15,3.24
3,21,ParlaMint-SI_2020-09-21-SDZ8-Redna-20.ana.u276,ParlaMint-SI_2020-09-21-SDZ8-Redna-20.ana.seg9...,"Hvala, podpredsednik. Hvala tudi za vprašanje,...","Hvala, podpredsednik.","Thank you, Vice President.",False,Negative,0.0,3.971972,{'Text_ID': 'ParlaMint-SI-en_2020-09-21-SDZ8-R...,32,1.98
4,53,ParlaMint-SI_2021-10-18-SDZ8-Redna-26.ana.u273,ParlaMint-SI_2021-10-18-SDZ8-Redna-26.ana.seg9...,"Kompliment na koncu je nepotreben, ker imava r...","Kompliment na koncu je nepotreben, ker imava r...",The compliment at the end is unnecessary becau...,False,Negative,0.0,1.713008,{'Text_ID': 'ParlaMint-SI-en_2021-10-18-SDZ8-R...,19,1.88
...,...,...,...,...,...,...,...,...,...,...,...,...,...
543,8611,ParlaMint-SI_2021-03-04-SDZ8-Izredna-64.ana.u150,ParlaMint-SI_2021-03-04-SDZ8-Izredna-64.ana.se...,Najlepša hvala. Da zaključim na hitro. Oznaka ...,Najlepša hvala.,Thank you very much.,False,Negative,0.0,4.312339,{'Text_ID': 'ParlaMint-SI-en_2021-03-04-SDZ8-I...,23,2.22
544,8634,ParlaMint-SI_2013-11-19-SDZ6-Redna-19.ana.u346,ParlaMint-SI_2013-11-19-SDZ6-Redna-19.ana.seg9...,Spoštovani gospod Tanko! Dolžna sem odgovor. D...,Spoštovani gospod Tanko!,Dear Mr. Tanko!,True,Negative,0.0,3.709246,{'Text_ID': 'ParlaMint-SI-en_2013-11-19-SDZ6-R...,9,2.48
545,8643,ParlaMint-SI_2020-11-16-SDZ8-Izredna-48.ana.u284,ParlaMint-SI_2020-11-16-SDZ8-Izredna-48.ana.se...,"Hvala za besedo, gospod podpredsednik. Hvala t...","Hvala za besedo, gospod podpredsednik.","Thank you for your word, Mr. Vice President.",False,Negative,0.0,3.626231,{'Text_ID': 'ParlaMint-SI-en_2020-11-16-SDZ8-I...,9,2.33
546,8652,ParlaMint-SI_2014-03-03-SDZ6-Redna-22.ana.u242,ParlaMint-SI_2014-03-03-SDZ6-Redna-22.ana.seg8...,"Hvala za besedo, predsedujoča. Pozdrav vsem pr...","Hvala za besedo, predsedujoča.","Thank you for your word, Madam President.",False,M_Negative,1.0,3.710418,{'Text_ID': 'ParlaMint-SI-en_2014-03-03-SDZ6-R...,15,3.17


In [10]:
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(df_count['annotation_utt_score'], df_count['count_avg'])
mae

np.float64(1.2387591240875913)

### Word-based length avg

In [11]:
df1["words"] = df1["text_sent"].apply(lambda x: len(x.split()))

# Compute the sentence-length-weighted average for each utterance
word_weighted_avg = (
    df1.groupby("ID")
    .apply(lambda group: (group["annotation_sent"] * group["words"]).sum() / group["words"].sum())
    .reset_index(name="word_avg")
)

# Add the length_avg back to the filtered DataFrame
df1 = df1.merge(word_weighted_avg, on="ID")
df1.head()

/var/folders/l_/t5l44v1s7d9168_x8bm5ly0h0000gn/T/ipykernel_74627/3809290448.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df1.groupby("ID")


,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata,count,count_avg,words,word_avg
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.84,4,2.808230
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.0,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.84,11,2.808230
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,2.0,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.84,5,2.808230
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,3.49,2,3.265585
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.0,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,3.49,4,3.265585


In [12]:
df1["word_avg"] = df1["word_avg"].round(2)
df_word = df1.drop_duplicates(subset=['ID']).reset_index()
df_word.head()

,index,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata,count,count_avg,words,word_avg
0,0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.84,4,2.81
1,3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,3.49,2,3.27
2,6,ParlaMint-SI_2011-06-21-SDZ5-Redna-29.ana.u178,ParlaMint-SI_2011-06-21-SDZ5-Redna-29.ana.seg6...,"Hvala lepa. V bistvu se strinjam z vami, gospo...",Hvala lepa.,Thank you very much.,False,Positive,5.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-21-SDZ5-R...,15,3.24,2,3.05
3,21,ParlaMint-SI_2020-09-21-SDZ8-Redna-20.ana.u276,ParlaMint-SI_2020-09-21-SDZ8-Redna-20.ana.seg9...,"Hvala, podpredsednik. Hvala tudi za vprašanje,...","Hvala, podpredsednik.","Thank you, Vice President.",False,Negative,0.0,3.971972,{'Text_ID': 'ParlaMint-SI-en_2020-09-21-SDZ8-R...,32,1.98,2,1.76
4,53,ParlaMint-SI_2021-10-18-SDZ8-Redna-26.ana.u273,ParlaMint-SI_2021-10-18-SDZ8-Redna-26.ana.seg9...,"Kompliment na koncu je nepotreben, ker imava r...","Kompliment na koncu je nepotreben, ker imava r...",The compliment at the end is unnecessary becau...,False,Negative,0.0,1.713008,{'Text_ID': 'ParlaMint-SI-en_2021-10-18-SDZ8-R...,19,1.88,32,1.40


In [13]:
mae = mean_absolute_error(df_word['annotation_utt_score'], df_word['word_avg'])
mae

np.float64(1.051259124087591)

### Character-based length avg

In [14]:
df1["char_length"] = df1["text_sent"].apply(len)
df1.head()


,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata,count,count_avg,words,word_avg,char_length
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.84,4,2.81,22
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.0,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.84,11,2.81,71
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,2.0,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.84,5,2.81,29
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,3.49,2,3.27,11
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.0,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,3.49,4,3.27,26


In [15]:
char_avg = (
    df1.groupby('ID')
    .apply(lambda x: (x["annotation_sent"] * x["char_length"]).sum() / x["char_length"].sum())
    .reset_index(name='char_avg')
)

df1 = df1.merge(char_avg, on='ID')
df1.head()

/var/folders/l_/t5l44v1s7d9168_x8bm5ly0h0000gn/T/ipykernel_74627/3011496211.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df1.groupby('ID')


,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata,count,count_avg,words,word_avg,char_length,char_avg
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.84,4,2.81,22,2.803708
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.0,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.84,11,2.81,71,2.803708
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,2.0,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.84,5,2.81,29,2.803708
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,3.49,2,3.27,11,3.265325
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.0,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,3.49,4,3.27,26,3.265325


In [16]:
df_char = df1.drop_duplicates(subset=['ID']).reset_index()
df_char.head()


,index,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata,count,count_avg,words,word_avg,char_length,char_avg
0,0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,2.84,4,2.81,22,2.803708
1,3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,3.49,2,3.27,11,3.265325
2,6,ParlaMint-SI_2011-06-21-SDZ5-Redna-29.ana.u178,ParlaMint-SI_2011-06-21-SDZ5-Redna-29.ana.seg6...,"Hvala lepa. V bistvu se strinjam z vami, gospo...",Hvala lepa.,Thank you very much.,False,Positive,5.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-21-SDZ5-R...,15,3.24,2,3.05,11,3.044246
3,21,ParlaMint-SI_2020-09-21-SDZ8-Redna-20.ana.u276,ParlaMint-SI_2020-09-21-SDZ8-Redna-20.ana.seg9...,"Hvala, podpredsednik. Hvala tudi za vprašanje,...","Hvala, podpredsednik.","Thank you, Vice President.",False,Negative,0.0,3.971972,{'Text_ID': 'ParlaMint-SI-en_2020-09-21-SDZ8-R...,32,1.98,2,1.76,21,1.788266
4,53,ParlaMint-SI_2021-10-18-SDZ8-Redna-26.ana.u273,ParlaMint-SI_2021-10-18-SDZ8-Redna-26.ana.seg9...,"Kompliment na koncu je nepotreben, ker imava r...","Kompliment na koncu je nepotreben, ker imava r...",The compliment at the end is unnecessary becau...,False,Negative,0.0,1.713008,{'Text_ID': 'ParlaMint-SI-en_2021-10-18-SDZ8-R...,19,1.88,32,1.40,197,1.376532


In [17]:
df_char["char_avg"] = df_char["char_avg"].round(2)


In [18]:
mae = (df_char["annotation_utt_score"] - df_char["char_avg"]).abs().mean()

print(f"Mean Absolute Error (MAE): {mae}")

Mean Absolute Error (MAE): 1.0504562043795622


## Additional feature building

In [19]:
df1 = df1.drop(columns=['count_avg', 'char_avg'])
df1.head()

,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata,count,words,word_avg,char_length
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,4,2.81,22
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.0,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,11,2.81,71
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,2.0,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,5,2.81,29
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,2,3.27,11
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.0,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,4,3.27,26


In [20]:
aggregation = df1.groupby('ID')['annotation_sent'].agg(
    mean = 'mean',
    median = 'median'

)

df1 = df1.merge(aggregation, on='ID')
df1.head()

,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata,count,words,word_avg,char_length,mean,median
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,4,2.81,22,2.838881,2.636969
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.0,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,11,2.81,71,2.838881,2.636969
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,2.0,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,5,2.81,29,2.838881,2.636969
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,2,3.27,11,3.491818,3.340636
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.0,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,4,3.27,26,3.491818,3.340636


In [21]:
df1['Q1'] = df1.groupby('ID')['annotation_sent'].transform(lambda x: x.quantile(0.25))
df1['Q3'] = df1.groupby('ID')['annotation_sent'].transform(lambda x: x.quantile(0.75))
df1.head()

,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata,count,words,word_avg,char_length,mean,median,Q1,Q3
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,4,2.81,22,2.838881,2.636969,2.438690,3.138115
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.0,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,11,2.81,71,2.838881,2.636969,2.438690,3.138115
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,2.0,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,3,5,2.81,29,2.838881,2.636969,2.438690,3.138115
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,2,3.27,11,3.491818,3.340636,3.195431,3.712613
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.0,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3,4,3.27,26,3.491818,3.340636,3.195431,3.712613
